# Forex & Fixed Income Response: Ship Movement + AI Sentiment Analysis

Analyzing how currencies and bonds respond to:
- Ship traffic disruptions in Strait of Hormuz
- AI-analyzed geopolitical sentiment
- Oil exporter vs importer currency dynamics
- Flight to safety in fixed income

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import glob
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

plt.rcParams['figure.figsize'] = (20, 12)
plt.rcParams['font.size'] = 11

## 1. Load Data

In [ ]:
# Load shipping data
shipping_df = pd.read_csv('Data/Portwatch_Shipment_Data/arrivals-of-ships.csv')
shipping_df['DateTime'] = pd.to_datetime(shipping_df['DateTime'])
shipping_df.set_index('DateTime', inplace=True)
shipping_df['Total'] = shipping_df[['Container', 'Dry Bulk', 'General Cargo', 'Roll-on/roll-off', 'Tanker']].sum(axis=1)
shipping_df['MA_30'] = shipping_df['Total'].rolling(30).mean()
shipping_df['Disruption_Pct'] = ((shipping_df['Total'] - shipping_df['MA_30']) / shipping_df['MA_30']) * 100

# Load sentiment data
sentiment_daily = pd.read_csv('Data/crisis_sentiment_daily.csv')
sentiment_daily['date'] = pd.to_datetime(sentiment_daily['date'])
sentiment_daily_indexed = sentiment_daily.set_index('date')

# Load Forex data
forex = {}
forex_files = glob.glob('Data/Forex/*.parquet')
for file in forex_files:
    pair_name = file.split('/')[-1].replace('.parquet', '')
    df = pd.read_parquet(file)
    df.index = pd.to_datetime(df.index)
    forex[pair_name] = df['Close']

forex_df = pd.DataFrame(forex)

# Load Fixed Income data
fixed_income = {}
fi_files = glob.glob('Data/Fixed_Income/*.parquet')
for file in fi_files:
    ticker = file.split('/')[-1].replace('.parquet', '')
    df = pd.read_parquet(file)
    df.index = pd.to_datetime(df.index)
    fixed_income[ticker] = df['Close']

fi_df = pd.DataFrame(fixed_income)

print(f"Forex pairs: {list(forex_df.columns)}")
print(f"Fixed Income: {list(fi_df.columns)}")
print(f"Date range: {forex_df.index.min()} to {forex_df.index.max()}")

## 2. Prepare Analysis Data

In [ ]:
# Combine forex and fixed income
analysis_df = forex_df.join(fi_df, how='outer')
analysis_df = analysis_df.join(shipping_df[['Total', 'Disruption_Pct']], how='left')
analysis_df = analysis_df.join(sentiment_daily_indexed[['sentiment_mean', 'article_count']], how='left')

# Forward fill
analysis_df['Disruption_Pct'] = analysis_df['Disruption_Pct'].fillna(method='ffill')
analysis_df['sentiment_mean'] = analysis_df['sentiment_mean'].fillna(0)

# Calculate returns
for col in list(forex_df.columns) + list(fi_df.columns):
    analysis_df[f'{col}_Return'] = analysis_df[col].pct_change()

# Crisis indicators
analysis_df['Shipping_Crisis'] = (analysis_df['Disruption_Pct'] < -20).astype(int)
analysis_df['Sentiment_Crisis'] = (analysis_df['sentiment_mean'] < -0.5).astype(int)
analysis_df['Combined_Crisis'] = ((analysis_df['Shipping_Crisis'] == 1) | 
                                   (analysis_df['Sentiment_Crisis'] == 1)).astype(int)

print(f"Analysis DataFrame: {analysis_df.shape}")
print(f"Crisis days: {analysis_df['Combined_Crisis'].sum()}")

## 3. Forex Analysis: Oil Exporters vs Importers

In [ ]:
# Categorize forex pairs
forex_categories = {
    'Oil Exporters': ['USD_CAD_Oil_Exporter', 'USD_NOK_Gas_Exporter', 'AUD_USD_LNG_Exporter'],
    'Oil Importers': ['USD_JPY_Oil_Importer', 'USD_KRW_Oil_Importer', 'USD_INR_Oil_Importer'],
    'Strategic': ['USD_CNH_Strategic_Importer']
}

scenarios = {
    'Normal': (analysis_df['Combined_Crisis'] == 0),
    'Shipping Crisis': (analysis_df['Shipping_Crisis'] == 1),
    'Sentiment Crisis': (analysis_df['Sentiment_Crisis'] == 1),
    'Combined Crisis': ((analysis_df['Shipping_Crisis'] == 1) & (analysis_df['Sentiment_Crisis'] == 1))
}

forex_results = {}
for category, pairs in forex_categories.items():
    available_pairs = [p for p in pairs if p in forex_df.columns]
    if not available_pairs:
        continue
    
    forex_results[category] = {}
    for scenario_name, mask in scenarios.items():
        return_cols = [f'{p}_Return' for p in available_pairs]
        avg_return = analysis_df[mask][return_cols].mean().mean() * 252 * 100
        forex_results[category][scenario_name] = avg_return

forex_results_df = pd.DataFrame(forex_results).T
print("\nForex Performance by Category and Scenario (Annualized %):")
print(forex_results_df.round(2))

In [ ]:
# Visualize forex performance
fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Plot 1: Returns by category
forex_results_df.T.plot(kind='bar', ax=axes[0], width=0.8)
axes[0].set_title('Forex Returns: Oil Exporters vs Importers', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Annualized Return (%)')
axes[0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0].legend(title='Category')
axes[0].grid(True, alpha=0.3)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# Plot 2: Crisis impact
forex_impact = forex_results_df.subtract(forex_results_df['Normal'], axis=0).drop('Normal', axis=1)
forex_impact.T.plot(kind='bar', ax=axes[1], width=0.8)
axes[1].set_title('Forex Crisis Impact (vs Normal)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Return Difference (%)')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1].legend(title='Category')
axes[1].grid(True, alpha=0.3)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 4. Fixed Income Analysis: Flight to Safety

In [ ]:
# Categorize fixed income
fi_categories = {
    'US Treasuries': ['TLT', 'IEF', 'SHY'],
    'Investment Grade': ['LQD'],
    'High Yield': ['HYG'],
    'Emerging Markets': ['EMB', 'EMLC']
}

fi_results = {}
for category, tickers in fi_categories.items():
    available_tickers = [t for t in tickers if t in fi_df.columns]
    if not available_tickers:
        continue
    
    fi_results[category] = {}
    for scenario_name, mask in scenarios.items():
        return_cols = [f'{t}_Return' for t in available_tickers]
        avg_return = analysis_df[mask][return_cols].mean().mean() * 252 * 100
        avg_vol = analysis_df[mask][return_cols].mean(axis=1).std() * np.sqrt(252) * 100
        fi_results[category][scenario_name] = avg_return
        fi_results[category][f'{scenario_name}_Vol'] = avg_vol

fi_results_df = pd.DataFrame(fi_results).T
return_cols = [c for c in fi_results_df.columns if '_Vol' not in c]
print("\nFixed Income Performance by Category (Annualized %):")
print(fi_results_df[return_cols].round(2))

In [ ]:
# Visualize fixed income performance
fig, axes = plt.subplots(2, 2, figsize=(20, 14))

# Plot 1: Returns by category
fi_results_df[return_cols].T.plot(kind='bar', ax=axes[0, 0], width=0.8)
axes[0, 0].set_title('Fixed Income Returns by Category', fontsize=14, fontweight='bold')
axes[0, 0].set_ylabel('Annualized Return (%)')
axes[0, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 0].legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].set_xticklabels(axes[0, 0].get_xticklabels(), rotation=45, ha='right')

# Plot 2: Crisis impact
fi_impact = fi_results_df[return_cols].subtract(fi_results_df['Normal'], axis=0).drop('Normal', axis=1)
fi_impact.T.plot(kind='bar', ax=axes[0, 1], width=0.8)
axes[0, 1].set_title('Fixed Income Crisis Impact (vs Normal)', fontsize=14, fontweight='bold')
axes[0, 1].set_ylabel('Return Difference (%)')
axes[0, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[0, 1].legend(title='Category', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].set_xticklabels(axes[0, 1].get_xticklabels(), rotation=45, ha='right')

# Plot 3: Individual bond ETF performance
bond_perf = {}
for ticker in fi_df.columns:
    bond_perf[ticker] = {}
    for scenario_name, mask in scenarios.items():
        bond_perf[ticker][scenario_name] = analysis_df[mask][f'{ticker}_Return'].mean() * 252 * 100

bond_perf_df = pd.DataFrame(bond_perf).T
bond_perf_df[return_cols].plot(kind='bar', ax=axes[1, 0], width=0.8)
axes[1, 0].set_title('Individual Bond ETF Performance', fontsize=14, fontweight='bold')
axes[1, 0].set_ylabel('Annualized Return (%)')
axes[1, 0].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
axes[1, 0].legend(title='Scenario')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=45, ha='right')

# Plot 4: Flight to safety (Treasuries vs High Yield spread)
if 'TLT' in fi_df.columns and 'HYG' in fi_df.columns:
    spread_data = {}
    for scenario_name, mask in scenarios.items():
        tlt_ret = analysis_df[mask]['TLT_Return'].mean() * 252 * 100
        hyg_ret = analysis_df[mask]['HYG_Return'].mean() * 252 * 100
        spread_data[scenario_name] = {'TLT': tlt_ret, 'HYG': hyg_ret, 'Spread': tlt_ret - hyg_ret}
    
    spread_df = pd.DataFrame(spread_data).T
    spread_df[['TLT', 'HYG']].plot(kind='bar', ax=axes[1, 1], width=0.8)
    axes[1, 1].set_title('Flight to Safety: Treasuries vs High Yield', fontsize=14, fontweight='bold')
    axes[1, 1].set_ylabel('Annualized Return (%)')
    axes[1, 1].axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    axes[1, 1].legend(['US Treasuries (TLT)', 'High Yield (HYG)'])
    axes[1, 1].grid(True, alpha=0.3)
    axes[1, 1].set_xticklabels(axes[1, 1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Calculate correlations
all_assets = list(forex_df.columns) + list(fi_df.columns)
corr_data = {}

for asset in all_assets:
    return_col = f'{asset}_Return'
    ship_corr = analysis_df[[return_col, 'Disruption_Pct']].corr().iloc[0, 1]
    sent_corr = analysis_df[[return_col, 'sentiment_mean']].corr().iloc[0, 1]
    
    corr_data[asset] = {
        'Shipping_Disruption': ship_corr,
        'Sentiment': sent_corr
    }

corr_df = pd.DataFrame(corr_data).T

print("\nCorrelations with Disruption and Sentiment:")
print(corr_df.round(3))

# Visualize
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr_df.T, annot=True, fmt='.3f', cmap='RdYlGn', center=0, 
            cbar_kws={'label': 'Correlation'}, ax=ax)
ax.set_title('Correlation: Forex & Fixed Income vs Disruption/Sentiment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Key Insights

In [ ]:
print("\n" + "="*80)
print("KEY INSIGHTS - FOREX & FIXED INCOME RESPONSE")
print("="*80)

print("\n1. FOREX: OIL EXPORTERS VS IMPORTERS")
print("   During Combined Crisis:")
for category in forex_results_df.index:
    normal = forex_results_df.loc[category, 'Normal']
    crisis = forex_results_df.loc[category, 'Combined Crisis']
    impact = crisis - normal
    print(f"   {category}: {crisis:+.2f}% (Impact: {impact:+.2f}%)")

print("\n2. FIXED INCOME: FLIGHT TO SAFETY")
print("   During Combined Crisis:")
for category in fi_results_df.index:
    normal = fi_results_df.loc[category, 'Normal']
    crisis = fi_results_df.loc[category, 'Combined Crisis']
    impact = crisis - normal
    print(f"   {category}: {crisis:+.2f}% (Impact: {impact:+.2f}%)")

if 'TLT' in bond_perf_df.index and 'HYG' in bond_perf_df.index:
    print("\n3. TREASURY vs HIGH YIELD SPREAD")
    for scenario in return_cols:
        tlt = bond_perf_df.loc['TLT', scenario]
        hyg = bond_perf_df.loc['HYG', scenario]
        spread = tlt - hyg
        print(f"   {scenario}: {spread:+.2f}% (TLT: {tlt:+.2f}%, HYG: {hyg:+.2f}%)")

print("\n4. STRONGEST CORRELATIONS WITH SHIPPING DISRUPTION:")
top_corr = corr_df['Shipping_Disruption'].abs().nlargest(5)
for asset, corr_val in top_corr.items():
    actual = corr_df.loc[asset, 'Shipping_Disruption']
    print(f"   {asset}: {actual:+.3f}")

print("\n5. TRADING IMPLICATIONS:")
print("   During Shipping + Sentiment Crisis:")
print("   - LONG: US Treasuries (TLT, IEF), Oil Exporter Currencies")
print("   - SHORT: High Yield Bonds (HYG), EM Bonds (EMB), Oil Importer Currencies")
print("   - HEDGE: Increase duration exposure, reduce credit risk")

print("\n" + "="*80)